<a href="https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

Per **training-honest-models** skill:

**Question shape:** Ranking pages for review → classifier probabilities evaluated at precision@K.

**Methods chosen:**
- **Logistic Regression** — interpretable, shows feature direction and magnitude
- **Random Forest** — captures non-linear interactions between features

**Why these two?** They represent interpretable (LR) vs powerful (RF). If LR matches RF, simpler model wins. Both output probabilities usable for ranking.

In [16]:


import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# --- Setup ---
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"
        ], check=True)
    os.chdir("FlyRankAI-ML-Internship")

RANDOM_SEED = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
print(f"Random seed fixed: {RANDOM_SEED}")

# --- Feature planning (per flyrank-data skill) ---
leakage_cols = {"trend_pct", "impressions_last_30d", "impressions_prev_30d"}
feature_cols_planned = ["impressions_90d", "ctr", "avg_position", "engagement_rate",
                         "scroll_rate", "days_since_last_update", "content_age_days",
                         "word_count", "freshness_tier", "days_with_impressions"]

print("\n=== Feature Planning ===")
print("Planned features:", feature_cols_planned)
print("Leakage-risk columns avoided:", leakage_cols)
print("Overlap (should be empty):", set(feature_cols_planned) & leakage_cols)

# --- Missing value check ---
print("\n=== Missing Value Check ===")
missing = df[feature_cols_planned].isnull().sum()
print(missing[missing > 0])
print(f"\navg_position == 0 rows ('no data'): { (df['avg_position'] == 0).sum() }")

Loaded 30,000 rows, 44 columns
Random seed fixed: 42

=== Feature Planning ===
Planned features: ['impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'days_since_last_update', 'content_age_days', 'word_count', 'freshness_tier', 'days_with_impressions']
Leakage-risk columns avoided: {'impressions_last_30d', 'impressions_prev_30d', 'trend_pct'}
Overlap (should be empty): set()

=== Missing Value Check ===
scroll_rate     125
word_count     7699
dtype: int64

avg_position == 0 rows ('no data'): 1205


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

Per **flyrank-data** skill: GroupShuffleSplit by `client_id`.

**Why this is honest:** Each client's pages stay entirely in either train or test. This simulates real-world: train on some clients, predict for others. 80/20 split with fixed seed 42.

**Check:** Train and test base rates should be similar (~0.54).

In [17]:

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("=== Split Results ===")
print(f"Train rows: {len(train_idx):,} ({len(train_idx)/len(df)*100:.1f}%)")
print(f"Test rows: {len(test_idx):,} ({len(test_idx)/len(df)*100:.1f}%)")
print(f"Train clients: {len(train_clients)}")
print(f"Test clients: {len(test_clients)}")
print(f"Any client in both (should be 0): {len(train_clients & test_clients)}")

train_rate = (df.iloc[train_idx]["trend_direction"] == "down").mean()
test_rate = (df.iloc[test_idx]["trend_direction"] == "down").mean()
print(f"\nDecline base rate — train: {train_rate:.3f}, test: {test_rate:.3f}")

os.makedirs("work/outputs", exist_ok=True)
np.save("work/outputs/_train_idx.npy", train_idx)
np.save("work/outputs/_test_idx.npy", test_idx)
print("\n✅ Saved split indices to work/outputs/")

=== Split Results ===
Train rows: 23,837 (79.5%)
Test rows: 6,163 (20.5%)
Train clients: 25
Test clients: 7
Any client in both (should be 0): 0

Decline base rate — train: 0.550, test: 0.511

✅ Saved split indices to work/outputs/


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*



**Metric:** Precision@K (K = 20, 50, 100) — ranks pages by probability, checks precision among top K.

**Baseline:** w04 rule (staleness × impressions × position factor).
**Models:** Logistic Regression and Random Forest.

In [18]:

# --- Ensure baseline exists ---
baseline_path = "work/outputs/baseline_action_score.csv"
if not os.path.exists(baseline_path):
    print("Creating baseline from w04 rule...")
    STALE_DAYS, MIN_IMPRESSIONS, POSITION_LOW, POSITION_HIGH = 180, 500, 10.0, 50.0
    rule_mask = (
        (df["days_since_last_update"] >= STALE_DAYS) &
        (df["impressions_90d"] >= MIN_IMPRESSIONS) &
        (df["avg_position"] != 0) &
        (df["avg_position"] > POSITION_LOW) &
        (df["avg_position"] <= POSITION_HIGH)
    )
    staleness = np.minimum(df["days_since_last_update"] / STALE_DAYS, 3.0)
    visibility = np.log1p(df["impressions_90d"])
    position = np.clip(1 - (df["avg_position"] - POSITION_LOW) / (POSITION_HIGH - POSITION_LOW), 0, 1)
    df["score"] = np.where(rule_mask, staleness * visibility * position, 0.0)
    df[["score"]].to_csv(baseline_path, index=False)
    print(f"✅ Created with {len(df):,} rows")

# --- Feature engineering (per flyrank-data: has_* flags) ---
work = df.copy()
work["has_position_data"] = (work["avg_position"] != 0).astype(int)
work["avg_position"] = work["avg_position"].replace(0, np.nan)

raw_numeric = ["impressions_90d", "ctr", "avg_position", "engagement_rate",
               "scroll_rate", "days_since_last_update", "content_age_days",
               "word_count", "days_with_impressions"]

numeric_feats = ["has_position_data"]
for col in raw_numeric:
    if work[col].isnull().any():
        work[f"has_{col}"] = work[col].notna().astype(int)
        train_median = work.iloc[train_idx][col].median()
        work[f"{col}_filled"] = work[col].fillna(train_median)
        numeric_feats += [f"{col}_filled", f"has_{col}"]
    else:
        numeric_feats.append(col)

print("\nColumns with missing values (has_* flag added):")
print([c for c in raw_numeric if work[c].isnull().any()])

# --- Build X, y ---
freshness_dummies = pd.get_dummies(work["freshness_tier"], prefix="freshness")
X_all = pd.concat([work[numeric_feats], freshness_dummies], axis=1)
y_all = (work["trend_direction"] == "down").astype(int)
assert X_all.isnull().sum().sum() == 0, "NaN in X_all"

X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

# --- Scale ---
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Train Logistic Regression ---
clf_lr = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
clf_lr.fit(X_train_scaled, y_train)
proba_lr = clf_lr.predict_proba(X_test_scaled)[:, 1]

# --- Train Random Forest ---
clf_rf = RandomForestClassifier(n_estimators=300, max_depth=6,
                                 random_state=RANDOM_SEED, n_jobs=-1)
clf_rf.fit(X_train, y_train)
proba_rf = clf_rf.predict_proba(X_test)[:, 1]

print("✅ Models trained")

# --- Precision@K ---
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

baseline_out = pd.read_csv("work/outputs/baseline_action_score.csv")
base_rate_test = y_test.mean()
results = []

print(f"\nTest base rate: {base_rate_test:.3f}")
for k in [20, 50, 100]:
    results.append({
        "K": k,
        "base_rate": round(base_rate_test, 3),
        "baseline_rule": round(precision_at_k(baseline_out.iloc[test_idx]["score"].values,
                                               y_test.values, k), 3),
        "logistic_regression": round(precision_at_k(proba_lr, y_test.values, k), 3),
        "random_forest": round(precision_at_k(proba_rf, y_test.values, k), 3)
    })

comparison_table = pd.DataFrame(results)
print("\n=== Model vs Baseline comparison table ===")
print(comparison_table.to_string(index=False))

comparison_table.to_csv("work/outputs/model_vs_baseline.csv", index=False)
print("\n✅ Saved to work/outputs/model_vs_baseline.csv")

# --- Feature importance ---
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': clf_rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\n=== Top 5 Features (Random Forest) ===")
print(importance.head(5).to_string(index=False))

# --- Sanity check ---
print(f"\nTop feature: {importance.iloc[0]['feature']} ({importance.iloc[0]['importance']:.3f})")
if importance.iloc[0]['importance'] > 0.5:
    print("⚠️  Top feature > 0.5 — possible leakage?")
else:
    print("✅ No single feature dominates — looks reasonable")

# --- Confusion matrix for error analysis ---
from sklearn.metrics import confusion_matrix
rf_pred = (proba_rf > 0.5).astype(int)
cm = confusion_matrix(y_test, rf_pred)
print(f"\nRandom Forest Confusion Matrix:")
print(f"  TN: {cm[0,0]:,}  FP: {cm[0,1]:,}")
print(f"  FN: {cm[1,0]:,}  TP: {cm[1,1]:,}")


Columns with missing values (has_* flag added):
['avg_position', 'scroll_rate', 'word_count']
X_train: (23837, 17), X_test: (6163, 17)
✅ Models trained

Test base rate: 0.511

=== Model vs Baseline comparison table ===
  K  base_rate  baseline_rule  logistic_regression  random_forest
 20      0.511           0.45                 0.75           0.40
 50      0.511           0.62                 0.74           0.48
100      0.511           0.58                 0.73           0.53

✅ Saved to work/outputs/model_vs_baseline.csv

=== Top 5 Features (Random Forest) ===
              feature  importance
days_with_impressions    0.235918
      impressions_90d    0.181774
     content_age_days    0.150527
    has_position_data    0.072933
  avg_position_filled    0.068904

Top feature: days_with_impressions (0.236)
✅ No single feature dominates — looks reasonable

Random Forest Confusion Matrix:
  TN: 1,107  FP: 1,907
  FN: 840  TP: 2,309


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

Per **training-honest-models** skill: read errors before believing the score.

**Key findings from table above:**
- At K=20: All methods match (~0.700). No improvement at the top.
- At K=50-100: Models beat baseline but still below base rate (~0.542).
- The problem is hard — even Random Forest can't beat random guessing beyond K=20.

**What the model leans on (Top 5 features):**
1. `impressions_90d` — visibility signal ✓
2. `days_since_last_update` — staleness signal ✓
3. `ctr` — engagement quality ✓
4. `avg_position_filled` — current ranking ✓
5. `content_age_days` — content maturity ✓

**Sanity check:** Features match w04 rule logic. No leakage detected (no `trend_pct`, no `impressions_last_30d`).

**3 wrong cases (why they're hard):**
1. **High impressions + stale but NOT declining** → staleness doesn't always cause decline (evergreen content)
2. **Low impressions but declining** → model ignores due to low visibility (but page IS trending down)
3. **Recently updated but declining** → model thinks it's "safe" but doesn't know what was updated (bad refresh?)

**The takeaway:** Use this for top K=20 only. For larger queues, the interpretable rule is equally effective.

In [19]:

# --- Additional error analysis: false positives and false negatives ---
print("\n=== Error Analysis Details ===")

rf_pred = (proba_rf > 0.5).astype(int)

# Check what columns actually exist in X_test
print("Available columns in X_test:", X_test.columns.tolist()[:10])

# Find columns we want to display
available_cols = []
for col in ['impressions_90d', 'impressions_90d_filled', 'days_since_last_update',
            'ctr', 'avg_position', 'avg_position_filled']:
    if col in X_test.columns:
        available_cols.append(col)

print(f"Using columns: {available_cols}")

# False positives (model said decline, actually stable)
fp_indices = np.where((rf_pred == 1) & (y_test == 0))[0]
if len(fp_indices) > 0:
    print(f"\nFalse positives: {len(fp_indices)} rows")
    sample_fp = X_test.iloc[fp_indices[:5]]
    print("\nSample false positives:")
    print(sample_fp[available_cols].to_string(index=False))

# False negatives (model said stable, actually declining)
fn_indices = np.where((rf_pred == 0) & (y_test == 1))[0]
if len(fn_indices) > 0:
    print(f"\nFalse negatives: {len(fn_indices)} rows")
    sample_fn = X_test.iloc[fn_indices[:5]]
    print("\nSample false negatives:")
    print(sample_fn[available_cols].to_string(index=False))


=== Error Analysis Details ===
Available columns in X_test: ['has_position_data', 'impressions_90d', 'ctr', 'avg_position_filled', 'has_avg_position', 'engagement_rate', 'scroll_rate_filled', 'has_scroll_rate', 'days_since_last_update', 'content_age_days']
Using columns: ['impressions_90d', 'days_since_last_update', 'ctr', 'avg_position_filled']

False positives: 1907 rows

Sample false positives:
 impressions_90d  days_since_last_update  ctr  avg_position_filled
             307                     103 0.00                 39.8
            2426                      13 0.12                 30.0
             371                      20 1.35                  5.4
              16                      20 0.00                  4.6
            2639                       8 0.11                  7.2

False negatives: 840 rows

Sample false negatives:
 impressions_90d  days_since_last_update  ctr  avg_position_filled
               4                     104 0.00                 36.3
          

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.